In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pydub

Mounted at /content/drive


In [3]:
import os
import glob
from pydub import AudioSegment

# 変換したいWAVが入っているGoogleドライブ内のフォルダパス
# （例: マイドライブ直下の「音声フォルダ」という名前の場合）
folder_path = "/content/drive/Shareddrives/音声メモ"

# フォルダ内の.wav / .WAV ファイルをすべて取得
wav_files = glob.glob(os.path.join(folder_path, "*.[wW][aA][vV]"))

if not wav_files:
    print("WAVファイルが見つかりません。パスを確認してください。")
else:
    for wav_path in wav_files:
        # 拡張子を.mp3に変更したファイルパスを作成
        mp3_path = os.path.splitext(wav_path)[0] + ".mp3"

        print(f"変換中: {os.path.basename(wav_path)} ...")

        # WAV読み込み -> MP3書き出し
        sound = AudioSegment.from_wav(wav_path)
        sound.export(mp3_path, format="mp3")

    print("すべての変換が完了しました！Googleドライブを確認してください。")

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


WAVファイルが見つかりません。パスを確認してください。


In [5]:
!pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 4.9 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=b8f7cbed5f0427e14a566f76ae1be6de1b09f50b6416626324148843bbe725bd
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper


In [8]:
import os
import glob
import whisper
from pydub import AudioSegment

# ==========================================
# 1. フォルダのパス設定（適宜書き換えてください）
# ==========================================
folder_path = "/content/drive/Shareddrives/音声メモ"

# ==========================================
# 2. WAVをMP3に変換し、元のWAVを削除する
# ==========================================
wav_files = glob.glob(os.path.join(folder_path, "*.[wW][aA][vV]"))

if wav_files:
    print(f"{len(wav_files)}件のWAVファイルを変換・削除します...")
    for wav_path in wav_files:
        mp3_path = os.path.splitext(wav_path)[0] + ".mp3"

        # MP3に変換
        print(f"変換中: {os.path.basename(wav_path)} -> MP3")
        sound = AudioSegment.from_wav(wav_path)
        sound.export(mp3_path, format="mp3")

        # 変換成功後、元のWAVファイルを削除
        os.remove(wav_path)
        print(f"削除完了: {os.path.basename(wav_path)}")
else:
    print("WAVファイルは見つかりませんでした（既にMP3のみの可能性があります）。")

# ==========================================
# 3. WhisperでMP3を一括文字起こし
# ==========================================
mp3_files = sorted(glob.glob(os.path.join(folder_path, "*.mp3")))

if not mp3_files:
    print("文字起こしするMP3ファイルがありません。")
else:
    print(f"\n{len(mp3_files)}件のMP3ファイルを文字起こしします...")

    # モデルの読み込み（日本語の精度を上げたい場合は "base" を "small" に変更）
    model = whisper.load_model("large-v3")
# モデルを large-v3 または large-v3-turbo に変更する
# base や small を使っている場合は精度が大幅に落ちます。ローカル環境やColabのGPUを使い、最強の精度を持つ large-v3（または高速な large-v3-turbo）を指定してください。
    all_transcriptions = []

    for mp3_path in mp3_files:
        filename = os.path.basename(mp3_path)
        print(f"文字起こし中: {filename} ...")

        # 文字起こしの実行
        result = model.transcribe(mp3_path)

        # どのファイルの音声かわかるように見出しをつけて保存
        all_transcriptions.append(f"■ {filename}\n{result['text']}\n")

    # ==========================================
    # 4. 1つのテキストファイルにまとめる（日付で命名）
    # ==========================================
    # 最初のファイル名（例: 2026-09-06_12-28-07.mp3）の先頭10文字（日付部分）を抽出
    first_filename = os.path.basename(mp3_files[0])
    date_str = first_filename[:10]

    # 保存先のパスを作成（例: 2026-09-06_文字起こし.txt）
    output_txt_path = os.path.join(folder_path, f"{date_str}_文字起こし.txt")

    # テキストファイルとして書き出し
    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(all_transcriptions))

    print(f"\nすべての処理が完了しました！\nテキストを保存しました: {output_txt_path}")

WAVファイルは見つかりませんでした（既にMP3のみの可能性があります）。

17件のMP3ファイルを文字起こしします...


100%|█████████████████████████████████████| 2.88G/2.88G [00:32<00:00, 95.4MiB/s]


文字起こし中: 2026-09-06_12-28-07.mp3 ...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


文字起こし中: 2026-09-06_20-55-46.mp3 ...


...

文字起こし中: 2026-09-07_20-11-23.mp3 ...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


文字起こし中: 2026-09-07_22-55-14.mp3 ...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



すべての処理が完了しました！
テキストを保存しました: /content/drive/Shareddrives/音声メモ/2026-09-06_文字起こし.txt
